# Road Following Live (JetRacer ONNX + ROS Subscriber Pipeline)

This notebook subscribes to the **ROS Camera Topic** (`/csi_cam_0/image_raw`) to avoid hardware CSI Camera device conflicts, executing ONNX model inference and **Stanley / PID Control** on the physical JetRacer.

### 1. Setup Environment & Load ONNX Model

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to sys.path to access Controller.py, Runner.py, and utils.py
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import onnxruntime as ort
try:
    from jetracer.utils import preprocess_onnx, bgr8_to_jpeg
except ImportError:
    from utils import preprocess_onnx, bgr8_to_jpeg

# Locate ONNX model file
model_path = os.path.join(Path.cwd(), "road_following_model.onnx")
if not os.path.exists(model_path):
    model_path = os.path.join(parent_dir, "notebooks", "road_following_model.onnx")

if not os.path.exists(model_path):
    print(f"[!] ERROR: ONNX model file '{model_path}' not found!")
else:
    print(f"[*] Loading ONNX model from: {model_path}")

available_providers = ort.get_available_providers()
providers = ['CUDAExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else []
providers.append('CPUExecutionProvider')

try:
    session = ort.InferenceSession(model_path, providers=providers)
except Exception:
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"[+] Loaded ONNX Session with providers: {session.get_providers()}")


### 2. Initialize ROS Node & JetRacer Hardware (`NvidiaRacecar`)

In [ ]:
import rospy
from sensor_msgs.msg import Image as ROSImage
from jetracer.nvidia_racecar import NvidiaRacecar
try:
    from jetracer.Controller import StanleyController, PIDController
    from jetracer.Runner import JetRacerROSOnnxRunner
except ImportError:
    from Controller import StanleyController, PIDController
    from Runner import JetRacerROSOnnxRunner

# 1. Initialize ROS Node
try:
    rospy.init_node('road_following_live_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

# 2. Hardware & Controller Setup
car = NvidiaRacecar()
stanley = StanleyController()
pid = PIDController()
stanley.reset()
pid.reset()

print("[+] JetRacer hardware and Stanley/PID Controllers initialized.")


### 3. Setup ROS Subscriber & ONNX Stanley Pipeline

In [ ]:
import cv2
import time
from IPython.display import display, Image, clear_output

# Default Parameters (Replacing argparse)
k_stanley = 2.5
base_throttle = 0.20
brake_gain = 0.10
steering_bias = 0.0
alpha = 0.4

latest_jpeg = None
latest_info = {}

def on_ros_frame(cv_image, raw_x, raw_y, smoothed_x, steering, dyn_throttle):
    global latest_jpeg, latest_info
    h, w = cv_image.shape[:2]
    px = int(w * (smoothed_x / 2.0 + 0.5))
    py = int(h * (raw_y / 2.0 + 0.5)) if raw_y != 0.0 else int(h * 0.5)

    prediction = cv_image.copy()
    cv2.circle(prediction, (px, py), 8, (0, 255, 0), 3)
    
    latest_jpeg = bgr8_to_jpeg(prediction)
    latest_info = {
        'raw_x': raw_x,
        'smoothed_x': smoothed_x,
        'steering': steering,
        'throttle': dyn_throttle
    }

runner = JetRacerROSOnnxRunner(
    session=session,
    input_name=input_name,
    output_name=output_name,
    car=car,
    stanley=stanley,
    k=k_stanley,
    throttle=base_throttle,
    brake_gain=brake_gain,
    bias=steering_bias,
    alpha=alpha,
    on_frame=on_ros_frame
)

runner.running = False  # Start paused

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, runner.image_callback, queue_size=1, buff_size=2**24)
print(f"[*] Subscribed to ROS Image Topic: {topic_name}")


### 4. Run Live Autonomous Driving & Stream Live Video Feed (Native Jupyter Stream)

In [ ]:
# Run this cell to START autonomous driving with live video stream!
# Interrupt/Stop kernel cell to stop the car safely.

runner.running = True
stanley.reset()
print("[+] AUTONOMOUS DRIVING ACTIVE!")

try:
    while True:
        if latest_jpeg is not None:
            clear_output(wait=True)
            print("=======================================================")
            print("   AUTONOMOUS DRIVING LIVE (ONNX + ROS Topic)          ")
            print(f"   Target X: {latest_info.get('raw_x', 0):+.3f} | Smoothed X: {latest_info.get('smoothed_x', 0):+.3f}")
            print(f"   Steering: {latest_info.get('steering', 0):+.3f} | Throttle: {latest_info.get('throttle', 0):.3f}")
            print("   (Interrupt/Stop kernel to stop the car)             ")
            print("=======================================================")
            display(Image(data=latest_jpeg, format='jpeg'))
        time.sleep(0.04)  # ~25 FPS stream refresh
except KeyboardInterrupt:
    pass
finally:
    runner.running = False
    car.throttle = 0.0
    car.steering = 0.0
    clear_output(wait=True)
    print("[+] Car STOPPED safely.")


### 5. Emergency Stop Cell

In [ ]:
# Emergency Stop
runner.running = False
car.throttle = 0.0
car.steering = 0.0
print("[+] Car STOPPED safely.")
